# Practical Work 2 — Data Analysis & Visualization

**Dataset:** `books_dataset.csv` (scraped from [books.toscrape.com](https://books.toscrape.com/))  
**Goal:** Analyse the books dataset using unsupervised learning (K-Means clustering) and then build an MLP classifier to predict the discovered genre clusters.  

**What we will do:**
1. Load and explore the dataset
2. Generate a word cloud from book titles
3. Prepare features for clustering
4. Find the optimal number of clusters (Elbow, Silhouette, Davies-Bouldin)
5. Assign genre labels and save the enriched dataset
6. Train MLP classifiers with different hyperparameters
7. Evaluate performance with confusion matrix and t-SNE

---
## Step 1 — Import Libraries

- **pandas / numpy** — data manipulation and math
- **matplotlib / seaborn** — visualizations
- **wordcloud** — generating the word cloud image
- **sklearn** — preprocessing, clustering, classification, evaluation, and t-SNE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

print('All libraries imported successfully!')

---
## Step 2 — Load and Explore the Dataset

We load `books_dataset.csv` which was scraped in the previous practical work.  
Quick check: shape, data types, first rows, basic statistics, and missing values.

In [ ]:
df = pd.read_csv('books_dataset.csv')

print(f'Shape: {df.shape}  ({df.shape[0]} books, {df.shape[1]} features)\n')
print('--- Data Types ---')
print(df.dtypes)
print('\n--- First 5 Rows ---')
df.head()

In [ ]:
print('--- Descriptive Statistics ---')
df.describe()

In [ ]:
print('--- Missing Values ---')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values!')

print('\n--- Category Distribution ---')
print(df['category'].value_counts())

---
## Step 3 — Word Cloud from Book Titles

We generate a word cloud from all the book titles to see which words appear the most.  
Common English stop-words ("the", "a", "of", etc.) are removed so the cloud highlights the meaningful keywords.

In [ ]:
all_titles = ' '.join(df['title'].astype(str).tolist()).lower()

stopwords = set(STOPWORDS)
stopwords.update(['one', 'two', 'three', 'new', 'us', 'book'])

wc = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    stopwords=stopwords,
    colormap='viridis',
    max_words=150,
    collocations=False,
)
wc.generate(all_titles)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Word Cloud of Book Titles', fontsize=20, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('wordcloud_titles.png', dpi=150)
plt.show()

In [ ]:
from collections import Counter

words = [w for w in all_titles.split() if w not in stopwords and len(w) > 2]
top_words = Counter(words).most_common(20)

print('Top 20 keywords in book titles:')
for word, count in top_words:
    print(f'  {word:20s} — {count}')

---
## Step 4 — Feature Engineering for Clustering

To cluster the books we need numerical features. Here is what we do:

1. **TF-IDF on titles** — turns each title into a vector of word importances (we keep the top 100 terms).
2. **Encode `category`** with LabelEncoder — turns the text category into an integer.
3. **Numeric columns** — `price` and `rating` are already numbers.
4. **Combine everything** into one feature matrix, then **scale** with StandardScaler.
5. **PCA** — reduce dimensionality so clustering is more efficient and avoids the curse of dimensionality.

In [ ]:
tfidf = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['title'].astype(str))
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

le_cat = LabelEncoder()
df['category_encoded'] = le_cat.fit_transform(df['category'].astype(str))

features = pd.concat([
    df[['price', 'rating', 'category_encoded']].reset_index(drop=True),
    tfidf_df.reset_index(drop=True),
], axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

pca = PCA(n_components=20, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Feature matrix shape after TF-IDF + numeric: {features.shape}')
print(f'After PCA (20 components): {X_pca.shape}')
print(f'Explained variance (20 PCs): {pca.explained_variance_ratio_.sum():.2%}')

---
## Step 5 — Finding the Optimal Number of Clusters

We run K-Means for k = 2 to 10 and evaluate three metrics:

| Metric | What it measures | Best value |
|--------|-----------------|------------|
| **Inertia (Elbow)** | Sum of squared distances to cluster center | Look for the "elbow" (bend) |
| **Silhouette Score** | How similar a point is to its own cluster vs neighbours | Higher is better (max 1) |
| **Davies-Bouldin Index** | Average similarity between each cluster and its most similar one | Lower is better |

In [ ]:
K_range = range(2, 11)
inertias = []
silhouettes = []
db_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels))
    db_scores.append(davies_bouldin_score(X_pca, labels))
    print(f'  k={k:2d}  |  Inertia={km.inertia_:10.1f}  |  Silhouette={silhouettes[-1]:.4f}  |  DB Index={db_scores[-1]:.4f}')

print('\nDone!')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(K_range, inertias, 'o-', color='#2196F3', linewidth=2, markersize=7)
axes[0].set_title('Elbow Method (Inertia)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_xticks(list(K_range))

axes[1].plot(K_range, silhouettes, 's-', color='#4CAF50', linewidth=2, markersize=7)
axes[1].set_title('Silhouette Score', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(list(K_range))

axes[2].plot(K_range, db_scores, 'D-', color='#FF5722', linewidth=2, markersize=7)
axes[2].set_title('Davies-Bouldin Index', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Number of Clusters (k)')
axes[2].set_ylabel('DB Index')
axes[2].set_xticks(list(K_range))

plt.tight_layout()
plt.savefig('cluster_metrics.png', dpi=150)
plt.show()

In [ ]:
best_k_sil = list(K_range)[np.argmax(silhouettes)]
best_k_db = list(K_range)[np.argmin(db_scores)]

print(f'Best k by Silhouette Score : {best_k_sil} (score = {max(silhouettes):.4f})')
print(f'Best k by Davies-Bouldin   : {best_k_db} (score = {min(db_scores):.4f})')

n_real_categories = df['category'].nunique()
optimal_k = max(best_k_sil, best_k_db, 5)
optimal_k = min(optimal_k, n_real_categories, 10)

print(f'\nDataset has {n_real_categories} unique categories.')
print(f'>>> We will use k = {optimal_k} for meaningful genre separation.')

---
## Step 6 — Apply Clustering & Assign Genre Labels

Now we run K-Means with the optimal k, add the cluster number to the dataframe,  
and map each cluster to a **Genre** name based on the most common `category` in that cluster.  
We also add a numeric **Genre_label** column.  
Finally, we save the enriched dataset as `books_clustered.csv`.

In [ ]:
km_final = KMeans(n_clusters=optimal_k, n_init=10, random_state=42)
df['cluster'] = km_final.fit_predict(X_pca)

used_genres = set()
cluster_to_genre = {}

for c in range(optimal_k):
    mask = df['cluster'] == c
    cats = df.loc[mask, 'category'].value_counts()
    chosen = None
    for cat_name in cats.index:
        if cat_name not in used_genres:
            chosen = cat_name
            break
    if chosen is None:
        chosen = f'{cats.index[0]}-{c}'
    cluster_to_genre[c] = chosen
    used_genres.add(chosen)
    print(f'  Cluster {c}: {mask.sum():3d} books  ->  Genre = "{chosen}"  (top category: {cats.index[0]})')

df['Genre'] = df['cluster'].map(cluster_to_genre)

le_genre = LabelEncoder()
df['Genre_label'] = le_genre.fit_transform(df['Genre'])

df.to_csv('books_clustered.csv', index=False)
print(f'\nDataset saved to books_clustered.csv  ({df.shape[0]} rows, {df.shape[1]} columns)')
df[['title', 'category', 'cluster', 'Genre', 'Genre_label']].head(10)

### 6.1 — Cluster Visualization (2D PCA)

We project the features down to 2 principal components and plot each book coloured by its genre cluster.

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=df['cluster'],
    cmap='Set2',
    alpha=0.7,
    edgecolors='white',
    linewidth=0.5,
    s=60,
)

handles = []
for c in sorted(df['cluster'].unique()):
    handles.append(plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=plt.cm.Set2(c / optimal_k),
                   markersize=10, label=f'{cluster_to_genre[c]} (cluster {c})'))
ax.legend(handles=handles, title='Genre (cluster)', fontsize=10, title_fontsize=11)

ax.set_title('Book Clusters by Genre (PCA 2D)', fontsize=16, fontweight='bold')
ax.set_xlabel(f'PC 1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC 2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.tight_layout()
plt.savefig('cluster_visualization.png', dpi=150)
plt.show()

---
## Step 7 — MLP Classification with Varying Hyperparameters

We use the **Genre_label** (from the clustering) as the target and train several MLP classifiers  
with different hyperparameter configurations:

- `hidden_layer_sizes` — architecture of the neural network
- `max_iter` — number of training epochs
- `learning_rate_init` — step size for the optimizer
- `activation` — activation function (relu, tanh)
- `solver` — optimizer (adam, sgd)

We compare each model's accuracy on the test set.

In [ ]:
y = df['Genre_label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set    : {X_test.shape[0]} samples')
print(f'Number of classes: {len(np.unique(y))}')

In [ ]:
configs = [
    {'hidden_layer_sizes': (50,),        'max_iter': 500,  'learning_rate_init': 0.001, 'activation': 'relu', 'solver': 'adam'},
    {'hidden_layer_sizes': (100,),       'max_iter': 500,  'learning_rate_init': 0.001, 'activation': 'relu', 'solver': 'adam'},
    {'hidden_layer_sizes': (100, 50),    'max_iter': 500,  'learning_rate_init': 0.001, 'activation': 'relu', 'solver': 'adam'},
    {'hidden_layer_sizes': (100, 100, 50), 'max_iter': 1000, 'learning_rate_init': 0.001, 'activation': 'relu', 'solver': 'adam'},
    {'hidden_layer_sizes': (100, 50),    'max_iter': 500,  'learning_rate_init': 0.01,  'activation': 'relu', 'solver': 'adam'},
    {'hidden_layer_sizes': (100, 50),    'max_iter': 500,  'learning_rate_init': 0.001, 'activation': 'tanh', 'solver': 'adam'},
    {'hidden_layer_sizes': (100, 50),    'max_iter': 1000, 'learning_rate_init': 0.001, 'activation': 'relu', 'solver': 'sgd'},
    {'hidden_layer_sizes': (100, 50),    'max_iter': 1000, 'learning_rate_init': 0.01,  'activation': 'tanh', 'solver': 'sgd'},
]

results = []
models = {}

for i, cfg in enumerate(configs, 1):
    mlp = MLPClassifier(random_state=42, **cfg)
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    models[i] = (mlp, y_pred)
    results.append({
        'Model': i,
        'hidden_layers': str(cfg['hidden_layer_sizes']),
        'max_iter': cfg['max_iter'],
        'lr': cfg['learning_rate_init'],
        'activation': cfg['activation'],
        'solver': cfg['solver'],
        'accuracy': round(acc, 4),
    })
    print(f'  Model {i}: accuracy = {acc:.4f}  |  layers={cfg["hidden_layer_sizes"]}  act={cfg["activation"]}  solver={cfg["solver"]}  lr={cfg["learning_rate_init"]}  epochs={cfg["max_iter"]}')

results_df = pd.DataFrame(results)
print('\n--- Results Summary ---')
results_df

---
## Step 8 — Evaluate the Best MLP Model

We pick the model with the highest test accuracy and show:

- **Classification report** — precision, recall, f1-score per class
- **Confusion matrix** — heatmap of true vs predicted labels
- **t-SNE visualization** — 2D embedding coloured by true and predicted labels

In [ ]:
best_idx = results_df['accuracy'].idxmax()
best_model_id = results_df.loc[best_idx, 'Model']
best_mlp, best_y_pred = models[best_model_id]

print(f'Best model: Model {best_model_id}  (accuracy = {results_df.loc[best_idx, "accuracy"]})')
print(f'Config: {configs[best_model_id - 1]}\n')

genre_names = le_genre.classes_
print('--- Classification Report ---')
print(classification_report(y_test, best_y_pred, target_names=genre_names))

### 8.1 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, best_y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=genre_names)
disp.plot(cmap='Blues', ax=ax, colorbar=True)
ax.set_title('Confusion Matrix — Best MLP Model', fontsize=15, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

### 8.2 — t-SNE Visualization

t-SNE reduces the high-dimensional features to 2D while preserving local neighbourhood structure.  
We plot two panels side-by-side: one coloured by **true labels** and one by **predicted labels** to visually check how well the classifier is doing.

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_test)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, labels, title in zip(
    axes,
    [y_test, best_y_pred],
    ['True Labels', 'Predicted Labels'],
):
    scatter = ax.scatter(
        X_tsne[:, 0], X_tsne[:, 1],
        c=labels,
        cmap='Set2',
        alpha=0.8,
        edgecolors='white',
        linewidth=0.5,
        s=60,
    )
    ax.set_title(f't-SNE — {title}', fontsize=14, fontweight='bold')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

    handles = []
    for lbl in sorted(np.unique(labels)):
        handles.append(plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor=plt.cm.Set2(lbl / max(np.unique(labels).max(), 1)),
                       markersize=10, label=genre_names[lbl]))
    ax.legend(handles=handles, fontsize=9, title='Genre')

plt.tight_layout()
plt.savefig('tsne_visualization.png', dpi=150)
plt.show()

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported all required libraries |
| 2 | Loaded `books_dataset.csv` — 500 books, 9 features |
| 3 | Generated a word cloud from book titles and found the top keywords |
| 4 | Built feature matrix (TF-IDF on titles + price, rating, category) and reduced with PCA |
| 5 | Ran K-Means for k=2..10, evaluated with Elbow, Silhouette, and Davies-Bouldin |
| 6 | Applied clustering with optimal k, assigned Genre labels, saved `books_clustered.csv` |
| 7 | Trained 8 MLP classifiers with different hyperparameters and compared accuracy |
| 8 | Evaluated best model: classification report, confusion matrix, and t-SNE visualization |